# 01 - მონაცემთა შეგროვება და გაერთიანება

ამ ნოუთბუქის მიზანია სამი წყაროდან მიღებული მონაცემების გაერთიანება და სუფთა dataset-ის შექმნა მოდელირებისთვის.

| ფაილი | წყარო | შინაარსი |
|-------|-------|----------|
| `players_data-2024_2025.csv` | FBref | სრული სტატისტიკა (277 სვეტი) |
| `players.csv` | Transfermarkt | დემოგრაფია, პოზიცია, სიმაღლე |
| `player_valuations.csv` | Transfermarkt | საბაზრო ღირებულების ისტორია |

**სამიზნე ცვლადი:** `market_value_in_eur`  
**სეზონები:** 2023/24, 2024/25  
**ლიგები:** Premier League, La Liga, Bundesliga, Serie A, Ligue 1

## 1. იმპორტები და კონფიგურაცია

In [16]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_DIR = '../data/raw/'

print('ბიბლიოთეკები ჩაიტვირთა')

ბიბლიოთეკები ჩაიტვირთა


## 2. ფაილების ჩატვირთვა

In [17]:
valuations = pd.read_csv(RAW_DIR + 'player_valuations.csv')
players    = pd.read_csv(RAW_DIR + 'players.csv')
fbref      = pd.read_csv(RAW_DIR + 'players_data-2024_2025.csv')

print(f"player_valuations : {valuations.shape[0]:,} სტრიქონი × {valuations.shape[1]} სვეტი")
print(f"players           : {players.shape[0]:,} სტრიქონი × {players.shape[1]} სვეტი")
print(f"fbref             : {fbref.shape[0]:,} სტრიქონი × {fbref.shape[1]} სვეტი")

player_valuations : 507,815 სტრიქონი × 6 სვეტი
players           : 47,716 სტრიქონი × 26 სვეტი
fbref             : 2,854 სტრიქონი × 267 სვეტი


## 3. უახლესი საბაზრო ღირებულება (2023 - 2025)

`player_valuations.csv` შეიცავს ღირებულების მთელ ისტორიას. ჩვენ გვჭირდება მხოლოდ **2023–2025 წლების უახლესი** ჩანაწერი თითო მოთამაშისთვის.

In [19]:
valuations['date'] = pd.to_datetime(valuations['date'])

latest_val = (valuations[
                  (valuations['date'] >= '2023-07-01') &
                  (valuations['date'] <= '2025-06-30')
              ]
              .sort_values('date')
              .groupby('player_id')
              .last()
              .reset_index()[['player_id', 'market_value_in_eur', 'date']])

print(f"მოთამაშეები 2023 - 2025 ღირებულებით: {len(latest_val):,}")
print(latest_val.head(3))

მოთამაშეები 2023 - 2025 ღირებულებით: 13,890
   player_id  market_value_in_eur       date
0       2857               100000 2023-08-22
1       3333              1000000 2025-05-30
2       3755               600000 2023-10-25


## 4. Big 5 ლიგის ფილტრაცია

მხოლოდ ევროპის ხუთი წამყვანი ლიგის მოთამაშეები, 2023 სეზონიდან.

| კოდი | ლიგა |
|------|------|
| GB1 | Premier League |
| ES1 | La Liga |
| L1 | Bundesliga |
| IT1 | Serie A |
| FR1 | Ligue 1 |

In [21]:
big5 = ['GB1', 'ES1', 'L1', 'IT1', 'FR1']

players_filtered = players[
    (players['current_club_domestic_competition_id'].isin(big5)) &
    (players['last_season'] >= 2023)
].copy()

players_val = players_filtered.merge(latest_val, on='player_id', how='inner')

print(f"Big 5 მოთამაშეები (2023+) : {len(players_filtered):,}")
print(f"ღირებულებასთან შეერთება  : {len(players_val):,}")

Big 5 მოთამაშეები (2023+) : 4,652
ღირებულებასთან შეერთება  : 3,620


## 5. სახელის სტანდარტიზაცია და გაერთიანება

FBref-სა და Transfermarkt-ს მოთამაშეების სახელები განსხვავებული ფორმატით აქვთ. სტანდარტიზაციის შემდეგ ვაერთებთ `_key` სვეტით.

In [22]:
def normalize(s):
    return (s.astype(str)
             .str.lower()
             .str.strip()
             .str.normalize('NFKD')
             .str.encode('ascii', errors='ignore')
             .str.decode('ascii')
             .str.replace(r'[^a-z\s]', '', regex=True)
             .str.replace(r'\s+', ' ', regex=True))

big5_fbref = ['eng Premier League', 'es La Liga', 'de Bundesliga',
              'it Serie A', 'fr Ligue 1']

fbref_filtered = fbref[fbref['Comp'].isin(big5_fbref)].copy()

players_val['_key'] = normalize(players_val['name'])
fbref_filtered['_key'] = normalize(fbref_filtered['Player'])

merged = fbref_filtered.merge(
    players_val[['_key', 'player_id', 'market_value_in_eur_y',
                 'position', 'sub_position', 'foot',
                 'height_in_cm', 'country_of_birth',
                 'date_of_birth', 'highest_market_value_in_eur', 'date']],
    on='_key', how='inner'
).drop(columns=['_key'])

merged = merged.rename(columns={'market_value_in_eur_y': 'market_value_in_eur'})

print(f"FBref (Big 5)  : {len(fbref_filtered):,} მოთამაშე")
print(f"Players + val  : {len(players_val):,} მოთამაშე")
print(f"გაერთიანება   : {len(merged):,} მოთამაშე")
print(f"სვეტები        : {merged.shape[1]}")

FBref (Big 5)  : 2,854 მოთამაშე
Players + val  : 3,620 მოთამაშე
გაერთიანება   : 2,111 მოთამაშე
სვეტები        : 277


## 6. შედეგის შემოწმება და შენახვა

In [23]:
print("── market_value_in_eur ──")
print(merged['market_value_in_eur'].describe().apply(lambda x: f'{x:,.0f}'))

print(f"\n── ლიგების განაწილება ──")
print(merged['Comp'].value_counts())

merged.to_csv(RAW_DIR + 'merged_raw.csv', index=False)
print(f"\n✓ შენახულია: data/raw/merged_raw.csv")
print(f"  {merged.shape[0]:,} მოთამაშე × {merged.shape[1]} სვეტი")
print("\n⏭ შემდეგი ეტაპი: 02_eda.ipynb")

── market_value_in_eur ──
count          2,111
mean      13,191,390
std       19,451,253
min           50,000
25%        2,000,000
50%        6,000,000
75%       18,000,000
max      180,000,000
Name: market_value_in_eur, dtype: object

── ლიგების განაწილება ──
Comp
it Serie A            483
eng Premier League    462
es La Liga            421
de Bundesliga         381
fr Ligue 1            364
Name: count, dtype: int64

✓ შენახულია: data/raw/merged_raw.csv
  2,111 მოთამაშე × 277 სვეტი

⏭ შემდეგი ეტაპი: 02_eda.ipynb
